# Update 1 — Same-condition comparison with the Mathar model

**Revision note (v2.0):** This notebook re-derives the comparison between the measured environmental coefficients and the Mathar model **on identical data rows**, replacing the earlier fixed-condition comparison that produced the retired "+17.2 % humidity enhancement" claim.

A linear surrogate model

$$n = n_0 + \alpha_T\,T + \alpha_H\,H + \alpha_P\,P$$

is fitted to (i) the measured refractive index and (ii) the Mathar formulation evaluated at the identical observed $(T,H,P)$ rows. Both fits share the same design matrix; the coefficient difference is estimated with a **paired moving-block bootstrap** that accounts for the correlation between the two estimates.

**Key numbers reproduced (cf. Supplemental Material, Table S2):**

| Analysis | N | $\Delta\alpha_T$ | $\Delta\alpha_H$ | $\Delta\alpha_P$ |
|---|---|---|---|---|
| In-domain (10–25 °C) | 12,134 | $-1.41\times10^{-8}$ | $+4.62\times10^{-9}$ | $+3.79\times10^{-9}$ |
| Full campaign | 145,784 | $-3.11\times10^{-9}$ | $+4.33\times10^{-9}$ | $+2.39\times10^{-9}$ |

Fraction of observations above 25 °C: 91.7 %. Block size L = 156 samples (residual ACF 1/e at $\tau$ = 78), B = 5000 replications.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf
import sys, os, time
sys.path.append('..')

from models.mathar.Mathar2007 import n as n_mathar_scalar

import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})

In [2]:
# Load data (columns: time, counts_ratio, humidity, temperature, pressure, n_1762)

df = pd.read_csv('../../data/processed/full_data.csv')
df = df.sort_values('time').reset_index(drop=True)   # ensure temporal order
print(f"N = {len(df)}")
print(df[['time','temperature','humidity','pressure','n_1762']].describe())

T_C = df['temperature'].values     # deg C
H_pct = df['humidity'].values      # %RH
P_hPa = df['pressure'].values      # hPa
n_data = df['n_1762'].values       # absolute refractive index


N = 145784
         temperature       humidity       pressure         n_1762
count  145784.000000  145784.000000  145784.000000  145784.000000
mean       28.874692      30.230868     986.403238       1.000254
std         3.380446       3.856421       4.481461       0.000003
min        17.518200      18.935562     964.605044       1.000248
25%        28.055459      27.315783     984.418391       1.000253
50%        29.446097      29.707467     987.300861       1.000254
75%        30.914016      33.085634     989.062836       1.000255
max        34.622549      45.320557    1006.774328       1.000270


In [3]:
# Mathar cloud evaluated at the identical (T, H, P) rows

lam_um = 1.762 # unit: µm
T_K = T_C + 273.15 # unit: K
P_Pa = P_hPa * 100.0 # unit: Pa

t0 = time.time()
n_mathar = np.array([n_mathar_scalar(lam_um, Tk, pp, hh)
                     for Tk, pp, hh in zip(T_K, P_Pa, H_pct)])
print(f"Mathar cloud generated in {time.time()-t0:.1f} s")
print(f"n_mathar range: [{n_mathar.min():.12f}, {n_mathar.max():.12f}]")


Mathar cloud generated in 1.6 s
n_mathar range: [1.000247321467, 1.000268634383]


In [4]:
# Analysis windows
mask_in  = (T_C >= 10.0) & (T_C <= 25.0)          # Mathar validity domain
mask_full = np.ones(len(df), dtype=bool)

n_above25 = int((~mask_in).sum())
print(f"In-domain (10-25 C): N = {mask_in.sum()}")
print(f"Above 25 C:          N = {n_above25}  ({100*n_above25/len(df):.1f} %)")


# Linear surrogate fit (OLS via normal equations; cross-checked vs statsmodels)
def fit_surrogate(idx):
    X = np.column_stack([np.ones(len(idx)), T_C[idx], H_pct[idx], P_hPa[idx]])
    beta, *_ = np.linalg.lstsq(X, n_data[idx], rcond=None)
    return beta                     # [n0, aT, aH, aP]

def fit_surrogate_mathar(idx):
    X = np.column_stack([np.ones(len(idx)), T_C[idx], H_pct[idx], P_hPa[idx]])
    beta, *_ = np.linalg.lstsq(X, n_mathar[idx], rcond=None)
    return beta

# Cross-check with statsmodels on the full sample
idx_full = np.arange(len(df))
Xf = np.column_stack([np.ones(len(df)), T_C, H_pct, P_hPa])
sm_res = sm.OLS(n_data, Xf).fit()
beta_manual = fit_surrogate(idx_full)
print("Sanity check (manual vs statsmodels, full sample):")
print("  manual  :", beta_manual)
print("  statsmodels:", sm_res.params)
print("  max abs diff:", np.abs(beta_manual - sm_res.params).max())


In-domain (10-25 C): N = 12134
Above 25 C:          N = 133650  (91.7 %)
Sanity check (manual vs statsmodels, full sample):
  manual  : [ 1.00002431e+00 -8.84742454e-07 -1.31524232e-08  2.59490560e-07]
  statsmodels: [ 1.00002431e+00 -8.84742454e-07 -1.31524232e-08  2.59490560e-07]
  max abs diff: 2.531308496145357e-14


In [5]:
# Point estimates: data vs Mathar surrogate

idx_in = np.where(mask_in)[0]

def report(name, bd, bm):
    print(f"--- {name} ---")
    print(f"  data   :  aT={bd[1]:.10e}  aH={bd[2]:.10e}  aP={bd[3]:.10e}")
    print(f"  mathar :  aT={bm[1]:.10e}  aH={bm[2]:.10e}  aP={bm[3]:.10e}")
    print(f"  Delta  : daT={bd[1]-bm[1]:+.6e}  daH={bd[2]-bm[2]:+.6e}  daP={bd[3]-bm[3]:+.6e}")
    print(f"  Delta %: daT={100*(bd[1]-bm[1])/abs(bm[1]):+.2f}%  daH={100*(bd[2]-bm[2])/abs(bm[2]):+.2f}%  daP={100*(bd[3]-bm[3])/abs(bm[3]):+.2f}%")

report("Full campaign", fit_surrogate(idx_full), fit_surrogate_mathar(idx_full))
report("In-domain (10-25 C)", fit_surrogate(idx_in), fit_surrogate_mathar(idx_in))


--- Full campaign ---
  data   :  aT=-8.8474245433e-07  aH=-1.3152423152e-08  aP=2.5949055957e-07
  mathar :  aT=-8.8163622117e-07  aH=-1.7485824221e-08  aP=2.5709600246e-07
  Delta  : daT=-3.106233e-09  daH=+4.333401e-09  daP=+2.394557e-09
  Delta %: daT=-0.35%  daH=+24.78%  daP=+0.93%
--- In-domain (10-25 C) ---
  data   :  aT=-9.3015150391e-07  aH=-6.7936612090e-09  aP=2.6935203329e-07
  mathar :  aT=-9.1606809452e-07  aH=-1.1415687397e-08  aP=2.6555829221e-07
  Delta  : daT=-1.408341e-08  daH=+4.622026e-09  daP=+3.793741e-09
  Delta %: daT=-1.54%  daH=+40.49%  daP=+1.43%


In [6]:
# Residual ACF -> block size

beta_full = fit_surrogate(idx_full)
resid = n_data - Xf @ beta_full
tau = None
nlags = min(2000, len(resid)//4)
acf_vals = acf(resid, nlags=nlags)
for l in range(1, len(acf_vals)):
    if acf_vals[l] < 1/np.e:
        tau = l
        break
L = max(2*tau, 10)
print(f"ACF 1/e decorrelation lag tau = {tau}")
print(f"Block size L = {L}")
assert L == 156, "Block size should be 156 for this dataset"


ACF 1/e decorrelation lag tau = 78
Block size L = 156


In [7]:
# Paired moving-block bootstrap of the coefficient difference
# Pre-aggregated per-block moments -> fast vectorized resampling

def paired_block_bootstrap(y_data, y_mathar, L, B=5000, seed=42):
    """Bootstrap distribution of beta_data - beta_mathar.
    Returns array of shape (B, 4): columns [n0, aT, aH, aP]."""
    N = len(y_data)
    nb = N // L
    rng = np.random.default_rng(seed)

    # per-block moments
    W = np.zeros((nb, 4, 4))   # X'X per block
    Ud = np.zeros((nb, 4))     # X'y_data per block
    Um = np.zeros((nb, 4))     # X'y_mathar per block
    for b in range(nb):
        i0 = b*L
        Xb = np.column_stack([np.ones(L), T_C[i0:i0+L], H_pct[i0:i0+L], P_hPa[i0:i0+L]])
        W[b]  = Xb.T @ Xb
        Ud[b] = Xb.T @ y_data[i0:i0+L]
        Um[b] = Xb.T @ y_mathar[i0:i0+L]

    Wf = W.reshape(nb, 16)          # (nb,16)
    diffs = np.empty((B, 4))
    for i in range(B):
        idx = rng.integers(0, nb, size=nb)
        XtX = Wf[idx].sum(axis=0).reshape(4, 4)
        bd = np.linalg.solve(XtX, Ud[idx].sum(axis=0))
        bm = np.linalg.solve(XtX, Um[idx].sum(axis=0))
        diffs[i] = bd - bm
    return diffs

t0 = time.time()
d_full = paired_block_bootstrap(n_data, n_mathar, L, B=5000)
d_in   = paired_block_bootstrap(n_data[mask_in], n_mathar[mask_in], L, B=5000)
print(f"Bootstrap completed in {time.time()-t0:.0f} s")

def boot_report(name, diffs):
    print(f"--- {name} ---")
    for j, lab in enumerate(['aT','aH','aP']):
        lo, hi = np.percentile(diffs[:, j+1], [2.5, 97.5])
        print(f"  Delta_{lab}: point~mean {diffs[:,j+1].mean():+.4e}   95% CI [{lo:+.4e}, {hi:+.4e}]")

boot_report("In-domain", d_in)
boot_report("Full campaign", d_full)


Bootstrap completed in 0 s
--- In-domain ---
  Delta_aT: point~mean -1.5470e-08   95% CI [-5.5079e-08, +2.6088e-08]
  Delta_aH: point~mean +5.8345e-09   95% CI [-6.8566e-09, +2.1536e-08]
  Delta_aP: point~mean +3.8041e-09   95% CI [+1.0301e-10, +8.0513e-09]
--- Full campaign ---
  Delta_aT: point~mean -3.1660e-09   95% CI [-6.2554e-09, -1.4601e-10]
  Delta_aH: point~mean +4.3590e-09   95% CI [+1.7920e-09, +6.8833e-09]
  Delta_aP: point~mean +2.4024e-09   95% CI [+3.0589e-10, +4.4513e-09]


In [9]:
# Blocked out-of-sample validation (K=5 folds, contiguous blocks)

def blocked_cv(y, L, K=5, seed=42):
    N = len(y)
    nb = N // L
    rng = np.random.default_rng(seed)
    block_folds = rng.permutation(np.tile(np.arange(K), int(np.ceil(nb/K)))[:nb])
    rmse = []
    for k in range(K):
        test_blocks  = np.where(block_folds == k)[0]
        train_blocks = np.where(block_folds != k)[0]
        test_rows  = np.concatenate([np.arange(b*L, min((b+1)*L, N)) for b in test_blocks])
        train_rows = np.concatenate([np.arange(b*L, min((b+1)*L, N)) for b in train_blocks])
        Xtr = np.column_stack([np.ones(len(train_rows)), T_C[train_rows], H_pct[train_rows], P_hPa[train_rows]])
        Xte = np.column_stack([np.ones(len(test_rows)),  T_C[test_rows],  H_pct[test_rows],  P_hPa[test_rows]])
        beta, *_ = np.linalg.lstsq(Xtr, y[train_rows], rcond=None)
        res = y[test_rows] - Xte @ beta
        rmse.append(np.sqrt(np.mean(res**2)))
    return np.array(rmse)

rmse_data   = blocked_cv(n_data, L)
rmse_mathar = blocked_cv(n_mathar, L)
print(f"Data surrogate CV RMSE   : {rmse_data.mean():.4e}  +/- {rmse_data.std():.2e}")
print(f"Mathar surrogate CV RMSE : {rmse_mathar.mean():.4e}  +/- {rmse_mathar.std():.2e}")


Data surrogate CV RMSE   : 1.8386e-07  +/- 1.76e-09
Mathar surrogate CV RMSE : 4.4407e-08  +/- 1.55e-09
